# Imports

In [1]:
#@title Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import ConcatDataset

import pydicom

import math
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import trange, tqdm
from pathlib import Path
from tqdm import tqdm
from PIL import Image

%matplotlib inline

# Reproducibility

In [2]:
#@title Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = True
# torch.use_deterministic_algorithms(True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


# EDA

In [3]:
img_path = "/kaggle/input/datasets/imbikramsaha/hindi-mnist/Hindi-MNIST/train/0/103265.png"

image = Image.open(img_path)

img_array = np.array(image)

print("Shape:", img_array.shape)
print("Dtype:", img_array.dtype)
print("Min pixel:", img_array.min())
print("Max pixel:", img_array.max())

Shape: (32, 32)
Dtype: uint8
Min pixel: 0
Max pixel: 255


# Preprocessing

In [4]:
ROOT_TRAIN = Path("/kaggle/input/datasets/imbikramsaha/hindi-mnist/Hindi-MNIST/train")
ROOT_TEST = Path("/kaggle/input/datasets/imbikramsaha/hindi-mnist/Hindi-MNIST/test")

In [5]:
class IVPTestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = sorted([
            os.path.join(root_dir, img)
            for img in os.listdir(root_dir)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, img_path

In [6]:
# TRANSFORMS
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),

    transforms.Resize((28, 28)),
    transforms.RandomRotation(10),          
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),             
        scale=(0.9, 1.1)                  
    ),

    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [7]:
# TRAIN DATASET
train_dataset = datasets.ImageFolder(
    root=ROOT_TRAIN,
    transform=train_transform
)
train_iivp_dataset = datasets.ImageFolder(
    root="/kaggle/input/datasets/muichimon/ivp-test/train/train",
    transform=train_transform
)

# VALIDATION DATASET
val_dataset = datasets.ImageFolder(
    root=ROOT_TEST,
    transform=test_transform
)
val_iivp_dataset = datasets.ImageFolder(
    root="/kaggle/input/datasets/muichimon/ivp-test/val/val",
    transform=test_transform
)

# IVP TEST-PREDICTIONS IMAGES DATASET
ivp_test_dataset = IVPTestDataset("/kaggle/input/datasets/muichimon/ivp-test/test/test", transform=test_transform)

In [8]:
train_loader = DataLoader(
    ConcatDataset([train_dataset, train_iivp_dataset]),
    batch_size=64,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    ConcatDataset([val_dataset, val_iivp_dataset]),
    batch_size=64,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    ivp_test_dataset, 
    batch_size=32, 
    shuffle=False,
    num_workers=2
)

# Model

In [9]:
model = models.resnet18(weights=None)  

model.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

model.fc = nn.Linear(model.fc.in_features, 10)

In [10]:
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train

In [11]:
def accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)
    return (predicted == labels).float().mean().item()

In [12]:
best_acc = 0
epochs_no_improve = 0
patience = 15

for epoch in range(100):  
    # -------------------
    # TRAIN
    # -------------------
    model.train()
    train_loss = 0
    train_acc = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_acc += accuracy(outputs, labels)

    train_loss /= len(train_loader)
    train_acc /= len(train_loader)

    # -------------------
    # TEST
    # -------------------
    model.eval()
    test_loss = 0
    test_acc = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            test_acc += accuracy(outputs, labels)

    test_loss /= len(val_loader)
    test_acc /= len(val_loader)

    # -------------------
    # PRINT
    # -------------------
    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f}, Test  Acc: {test_acc:.4f}")
    print("-"*40)

    # -------------------
    # EARLY STOPPING
    # -------------------
    if test_acc > best_acc:
        best_acc = test_acc
        epochs_no_improve = 0
    
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_test_acc": best_acc
        }, "/kaggle/working/best_checkpoint.pth")
    
        print(f"New best model saved! Acc: {best_acc:.4f}")
    
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

Epoch 1
Train Loss: 0.2064, Train Acc: 0.9334
Test  Loss: 0.0273, Test  Acc: 0.9909
----------------------------------------
New best model saved! Acc: 0.9909
Epoch 2
Train Loss: 0.0675, Train Acc: 0.9794
Test  Loss: 0.0211, Test  Acc: 0.9945
----------------------------------------
New best model saved! Acc: 0.9945
Epoch 3
Train Loss: 0.0444, Train Acc: 0.9865
Test  Loss: 0.0151, Test  Acc: 0.9958
----------------------------------------
New best model saved! Acc: 0.9958
Epoch 4
Train Loss: 0.0392, Train Acc: 0.9879
Test  Loss: 0.0149, Test  Acc: 0.9958
----------------------------------------
Epoch 5
Train Loss: 0.0299, Train Acc: 0.9913
Test  Loss: 0.0192, Test  Acc: 0.9953
----------------------------------------
Epoch 6
Train Loss: 0.0305, Train Acc: 0.9914
Test  Loss: 0.0124, Test  Acc: 0.9962
----------------------------------------
New best model saved! Acc: 0.9962
Epoch 7
Train Loss: 0.0313, Train Acc: 0.9910
Test  Loss: 0.0079, Test  Acc: 0.9983
------------------------------

# TEST (Prediction Submissions)

## LOAD MODEL

In [13]:
checkpoint = torch.load("/kaggle/input/datasets/muichimon/checkpoints/model_weight_checkpoints/hindi-mnist/resnet18_best_v2.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## EVALUATE

In [14]:
model.eval()

predictions = []

with torch.no_grad():
    for images, paths in test_loader:
        images = images.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        for path, pred in zip(paths, preds):
            predictions.append((path, pred.item()))

## SAVE SUBMISSIONS INTO CSV

In [15]:
import os
import pandas as pd

formatted_preds = []

for path, pred in predictions:
    filename = os.path.basename(path)        
    img_id = os.path.splitext(filename)[0]   
    formatted_preds.append((int(img_id), pred))

# Create DataFrame
df = pd.DataFrame(formatted_preds, columns=["Id", "Category"])

# Sort by Id (important for consistency)
df = df.sort_values(by="Id")

# Save CSV
df.to_csv("submission.csv", index=False)